In [2]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list
import Osteosarcoma.Data_processing as Data_processing

from __future__ import annotations

import os  # needed navigate the system to get the input data

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### patchify image

In [2]:
patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx'
build = Build_list.Build(patient_list_file)
batch_list, patient_set_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')
# show one example of the image and mask
print('patient set:', patient_set_list[0], 'patient index:', patient_index_list[0], 'label:', label_list[0], 'image path:', image_path_list[0], 'mask path:', mask_path_list[0])


data_path = '/host/e/D/Data/Habitats/Jishuitan/original_data'


Number of cases to process: 330
patient set: set_1 patient index: 1 label: 0 image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/img.nii.gz mask path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label.nii.gz


In [3]:
for index in range(0,len(patient_index_list)):
    patient_set = patient_set_list[index]
    patient_index = patient_index_list[index]

    # if already have patch_table.xlsx, skip
    patch_table_path = os.path.join('/host/e/D/Data/Habitats/Jishuitan/habitats/', str(patient_set), str(patient_index), "patch_table.xlsx")
    if os.path.exists(patch_table_path):
        print(f"Patch table already exists for patient {patient_index}, skipping...")
        continue
  
    image_path = os.path.join(data_path, str(patient_set),str(patient_index), 'img.nii.gz')
    label_path = os.path.join(data_path, str(patient_set),str(patient_index), 'label.nii.gz')
    print('image path:', image_path, '\nlabel path:', label_path)

    habitats_out_path = '/host/e/D/Data/Habitats/Jishuitan/habitats/'
    ff.make_folder([os.path.join(habitats_out_path, str(patient_set)), os.path.join(habitats_out_path, str(patient_set), str(patient_index))])
    
    label_nii = nb.load(label_path)
    label_arr = label_nii.get_fdata()
    affine = label_nii.affine; header = label_nii.header

    # bbox
    bbox_out_path = os.path.join(habitats_out_path, str(patient_set), str(patient_index), 'tumor_bbox.nii.gz')
    bbox_arr, x0, x1, y0, y1, z0, z1 = Data_processing.bbox3d(label_arr, buffer_x=0, buffer_y = 0, buffer_z = 0 )
    bbox_nii = nb.Nifti1Image(bbox_arr, affine=affine, header=header)
    nb.save(bbox_nii, bbox_out_path)

    # # extract patches

    patch_size_mm = 10.0               # 1 cm
    min_tumor_fraction = 0.1          # almost include everyone, will exclude based on table later
    out_dir = os.path.join(habitats_out_path, str(patient_set), str(patient_index), "patches")
    os.makedirs(out_dir, exist_ok=True)

    patch_table_path = os.path.join(habitats_out_path, str(patient_set), str(patient_index), "patch_table.xlsx")

    # 你上一步已经得到 bbox_arr（shape = (X,Y,Z)），以及 bbox 的坐标 x0,x1,y0,y1,z0,z1
    # 这里假设你已经有：bbox_arr, x0,x1,y0,y1,z0,z1
    # 并且 label_path 指向 label.nii.gz

    count_included, total_patches = Data_processing.patchify(label_arr, label_nii, x0, x1, y0, y1, z0, z1, patch_size_mm, min_tumor_fraction, out_dir, patch_table_path)
    print(f'Patient {patient_index}: Included patches {count_included} out of {total_patches} total patches.')

    # ## assert 
    # patch_file_list = ff.find_all_target_files(['*'],out_dir)
    # patch_collection = np.zeros_like(label_arr)
    # for jj in range(0, len(patch_file_list)):
    #     patch_nii = nb.load(patch_file_list[jj])
    #     patch_arr = patch_nii.get_fdata()
    #     patch_collection += patch_arr

    # # assert patch collection should only have 2 unique values, [0,1]
    # unique_values = np.unique(patch_collection)
    # assert np.array_equal(unique_values, [0, 1]), f"Patch collection has unexpected values: {unique_values}"
    
    # # save patch collection
    # patch_collection_nii = nb.Nifti1Image(patch_collection, affine=affine, header=header)
    # patch_collection_path = os.path.join(habitats_out_path, str(patient_set), str(patient_index), "patch_collection.nii.gz")
    # nb.save(patch_collection_nii, patch_collection_path)

Patch table already exists for patient 1, skipping...
Patch table already exists for patient 5, skipping...
Patch table already exists for patient 7, skipping...
Patch table already exists for patient 8, skipping...
Patch table already exists for patient 11, skipping...
Patch table already exists for patient 15, skipping...
Patch table already exists for patient 18, skipping...
Patch table already exists for patient 19, skipping...
Patch table already exists for patient 20, skipping...
Patch table already exists for patient 21, skipping...
Patch table already exists for patient 22, skipping...
Patch table already exists for patient 23, skipping...
Patch table already exists for patient 24, skipping...
Patch table already exists for patient 26, skipping...
Patch table already exists for patient 28, skipping...
Patch table already exists for patient 29, skipping...
Patch table already exists for patient 30, skipping...
Patch table already exists for patient 33, skipping...
Patch table al

### initiate extractor

In [3]:
# Instantiate the extractor
paramPath = '/host/d/Github/Osteosarcoma/radiomics_settings/MR_setting_image.yaml'
extractor = featureextractor.RadiomicsFeatureExtractor(paramPath)

print('Extraction parameters:\n\t', extractor.settings)
print('Enabled filters:\n\t', extractor.enabledImagetypes)
print('Enabled features:\n\t', extractor.enabledFeatures)

Extraction parameters:
	 {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': True, 'normalizeScale': 100, 'removeOutliers': None, 'resampledPixelSpacing': [1, 1, 1], 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'binWidth': 25, 'voxelArrayShift': 300, 'geometryTolerance': 0.0001}
Enabled filters:
	 {'Original': {}, 'LoG': {'sigma': [2.0, 4.0]}, 'Wavelet': {}}
Enabled features:
	 {'shape': None, 'firstorder': None, 'glcm': ['Autocorrelation', 'JointAverage', 'ClusterProminence', 'ClusterShade', 'ClusterTendency', 'Contrast', 'Correlation', 'DifferenceAverage', 'DifferenceEntropy', 'DifferenceVariance', 'JointEnergy', 'JointEntropy', 'Imc1', 'Imc2', 'Idm', 'Idmn', 'Id', 'Idn', 'InverseVariance', 'MaximumProbability', 'SumEntropy', 'SumSquares'], 'glrlm': None, 'glszm': None, 'gldm': None, 'ngtdm': None}


### define patient list

In [5]:
patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx'
build = Build_list.Build(patient_list_file)
batch_list, patient_set_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')

Number of cases to process: 330


### extract patch features

In [7]:
data_path = '/host/e/D/Data/Habitats/Jishuitan/original_data'
rows = []
tumor_fraction_threshold = 0.8
for i in range(0,len(patient_index_list)):
    patient_set = patient_set_list[i]
    patient_index = patient_index_list[i]
    print('Processing patient set:', patient_set, 'patient index:', patient_index, ' i is ', i)
    img_p = os.path.join(data_path, str(patient_set), str(patient_index), 'img.nii.gz')
    # save_folder
    save_folder = os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_set), str(patient_index))
    ff.make_folder([os.path.dirname(save_folder), save_folder])

    # if os.path.exists(os.path.join(save_folder, 'radiomics_features_patches.xlsx')):
    #     print('  Features already extracted for this patient. Skipping...')
    #     continue
    
    rows = []

    # we need to extract features for each patch
    patches_dir = os.path.join('/host/e/D/Data/Habitats/Jishuitan/habitats/', str(patient_set), str(patient_index), 'patches')
    patch_table_raw = pd.read_excel(os.path.join('/host/e/D/Data/Habitats/Jishuitan/habitats/', str(patient_set), str(patient_index), 'patch_table.xlsx'))
    # only keep patches with ['tumor_fraction'] >= 0.8
    patch_table = patch_table_raw[patch_table_raw['tumor_fraction'] >= tumor_fraction_threshold].reset_index(drop=True)
    print(f'  Number of patches to process: {patch_table.shape[0]}', ' (out of ', patch_table_raw.shape[0], ' total patches )')

    for j in range(0, patch_table.shape[0]):
        patch_id = patch_table.loc[j, 'patch_id']
        tumor_fraction = patch_table.loc[j, 'tumor_fraction']
        patch_p = os.path.join(patches_dir, f'patch_{patch_id:04d}.nii.gz')
        print('  Processing patch:', patch_p)
        msk_p = patch_p  # mask is the same as image for patches
        result = extractor.execute(img_p,msk_p) # important!
        
        # Keep only radiomics features (drop diagnostics)
        feats = {k: v for k, v in result.items() if not k.startswith("diagnostics_")}
        feats["Patient_set"] = patient_set
        feats["Patient_index"] = patient_index
        feats["patch_id"] = patch_id
        feats["tumor_fraction"] = tumor_fraction
        feats["Image_filepath"] = img_p
        feats["Mask_filepath"] = msk_p
        
        rows.append(feats)
        df = pd.DataFrame(rows)
        front_cols = ["Patient_set", "Patient_index", "patch_id","tumor_fraction", "Image_filepath", "Mask_filepath"]
        other_cols = [col for col in df.columns if col not in front_cols]
        df = df[front_cols + other_cols]
        df.to_excel(os.path.join(save_folder, 'radiomics_features_patches.xlsx'), index=False)

Processing patient set: set_1 patient index: 1  i is  0
  Number of patches to process: 596  (out of  828  total patches )
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0004.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0005.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0006.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0009.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0010.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0011.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0012.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0015.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/1/patches/patch_0016.nii.gz
  Pr

/usr/local/lib/python3.10/dist-packages/radiomics/glcm.py:599: RuntimeWarning: invalid value encountered in sqrt
  imc2 = (1 - numpy.e ** (-2 * (HXY2 - HXY))) ** 0.5


  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0007.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0010.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0016.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0019.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0020.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0022.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0025.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0028.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0029.nii.gz
  Processing patch: /host/e/D/Data/Habitats/Jishuitan/habitats/set_1/74/patches/patch_0030.nii.gz
  Processing patch: 

KeyboardInterrupt: 

### normalize features

#### normalize features, will use the min and max values from whole-tumor 

In [9]:
scale_df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_features_list.xlsx')

patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx'
build = Build_list.Build(patient_list_file)
batch_list, patient_set_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')

for i in range(3,8):#len(patient_index_list)):
    patient_set = patient_set_list[i]
    patient_index = patient_index_list[i]


    feature_file = os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_set), str(patient_index), 'radiomics_features_patches.xlsx')
    if os.path.isfile(feature_file) == False:
        print(f'  Feature file not found for this patient. Skipping... {feature_file}')
        continue
    df = pd.read_excel(feature_file)
    
    # Normalize each feature
    for feature in scale_df['feature_name']:
        if feature in df.columns:
            min_val = scale_df.loc[scale_df['feature_name'] == feature, 'feature_min'].values[0]
            max_val = scale_df.loc[scale_df['feature_name'] == feature, 'feature_max'].values[0]
            # if max_val - min_val != 0:
            df[feature] = (df[feature] - min_val) / (max_val - min_val)
           
        else:
            print(f'Feature {feature} not found in dataframe for patient {patient_index}. Skipping normalization for this feature.')
    
    # Save the normalized features
    df.to_excel(os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_set), str(patient_index), 'radiomics_features_patches_normalized.xlsx'), index=False)

#     # only pick the selected features for Svm and save
#     df_svm = df[["Patient_index", "patch_id","tumor_fraction", "Image_filepath", "Mask_filepath"] + svm_selected_radiomics_features]
#     # for each feature, check if there are any values < -1 or > 2 and print the count
#     for feature in svm_selected_radiomics_features:
#         out_of_bounds = ((df_svm[feature] < -1) | (df_svm[feature] > 2)).sum()
#         if out_of_bounds > 0:
#             print(f'Feature {feature} has {out_of_bounds} out-of-bounds elements out of {df_svm.shape[0]} , percent: {out_of_bounds/df_svm.shape[0]*100:.2f}% for patient {patient_index}')
#     # out_of_bounds = ((df_svm[svm_selected_radiomics_features] < -1) | (df_svm[svm_selected_radiomics_features] > 2)).sum().sum()
#     # print(f'Number of out-of-bounds elements in SVM selected features for patient {patient_index}: {out_of_bounds}', 'out of', df_svm[svm_selected_radiomics_features].size, 'total elements')
#     df_svm.to_excel(os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'radiomics_features_patches_normalized_svm_selected.xlsx'), index=False)


Number of cases to process: 330
